#### Context Length Setting

Check context length of qwen3-0.6b, and longest len of my chunks

In [1]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-Embedding-0.6B')

In [ ]:
from custom_textsplitter import CustomTextSplitter
import pandas as pd

chat_df = pd.read_csv('data/processed_data/whatsapp_chats.csv')
splitter = CustomTextSplitter()

chunk_df = splitter.split_messages(message_df=chat_df,chunk_size=1000,return_df=True)
chunk_df['tokenized_chunks_len'] = chunk_df['chunk_text'].apply(tokenizer.encode).apply(len)

: 

In [11]:
chunk_df['tokenized_chunks_len'].describe()

count    5085.000000
mean      306.621042
std        59.305509
min         8.000000
25%       269.000000
50%       300.000000
75%       342.000000
max       691.000000
Name: tokenized_chunks_len, dtype: float64

: 

#### Enhance Metadata Filtering 

In [ ]:
import ast
import chromadb


## Add other_person name for pre-filtering chats in Chroma
client = chromadb.PersistentClient('.chroma_db')
collection = client.get_collection('chat_documents')


: 

In [ ]:

all_docs = collection.get(include=["metadatas"])
other_persons = []

for doc in all_docs['metadatas']:

    is_gc = doc['is_groupchat']
    if not is_gc:
        participants = ast.literal_eval(doc['participants'])
        other_person = [person for person in participants if person!='Dan'][0]
    else:
        other_person='N/A'

    doc['other_person'] = other_person
    ## separate thing add source as whatsapp
    doc['source'] = 'whatsapp'

    other_persons.append(doc)


collection.update(
    ids=all_docs['ids'],
    metadatas=other_persons
)

: 

In [ ]:
from datetime import datetime, date

metadatas = []

for doc in all_docs['metadatas']:
    date_range_str = doc['date_range']

    start_str, end_str = date_range_str.split(' - ')
    start_date = datetime.strptime(start_str.strip(), '%Y-%m-%d').date()
    end_date = datetime.strptime(end_str.strip(), '%Y-%m-%d').date()

    start_date=int(start_date.strftime("%Y%m%d"))
    end_date= int(end_date.strftime("%Y%m%d"))

    doc['start_date'] = start_date
    doc['end_date'] = end_date

    metadatas.append(doc)

collection.update(
    all_docs['ids'],
    metadatas=metadatas
)

: 

In [ ]:
filter_date = datetime.strptime("2023-02-01",'%Y-%m-%d')
filter_date = int(filter_date.strftime("%Y%m%d"))

documents = collection.query(
    "NonaSSA gc messages", 
    k=5, 
    filter={
        "$and": [
        {"start_date" : {"$lte" : filter_date}},
        {"other_person": "Damian"}
        ]})
first_document = documents[1] if documents else None
first_document.metadata

: 

#### Trial for the Reranker Model

In [ ]:
# Helper function for printing docs
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i + 1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

: 

In [2]:
from ollama_utils import get_vector_store

vs = get_vector_store()
retriever = vs.as_retriever(search_kwargs={"k": 20})

In [3]:
query = "What was done to Russia?"
docs = retriever.invoke(query)
pretty_print_docs(docs)

Document 1:

**Metadata of Conversation**
        Chat: McGregor one love (part 596) | Date: 2022-02-24 - 2022-03-06 | Language: ro
        **End of Metadata of Conversation** 
 siscanuud: Zguduieu ferestrele
siscanuud: De la bombe batalioane in Odessa
Iani 🛸: Nam auzit nici o pula
Iani 🛸: Da unde io futut o bomba ?
siscanuud: Huioznaet
siscanuud: Azi la ora 5 dimineata
Dragos 🔥🎸: Patani, am fututo in europa
Petru Leşenco: https://www.jurnal.md/ro/news/f767d65a463f2862/putin-ataca-ucraina-cu-avioane-tancuri-si-nave-militare-zelensky-declara-stare-de-razboi.html
Cristian: Salam vsem salam
Cristian: Ia poshol v shvetiu
Cristian: Uvidimsea
Cristian: Ii pizda dela
siscanuud: Batea cand m-am trezit neo zis
siscanuud: Eu tot
siscanuud: Tat normal brat
Teodor Lungu: I am futut un boșet
Teodor Lungu: Wai
siscanuud: Caroci am gasit un vidos vechi
siscanuud: Si i-am futut un tiktok
siscanuud: Vreti sa planjeti oleaq
siscanuud: Patani
siscanuud: Petru inca era lisii suqa😂😂😂
Iani 🛸: 😂😂😂😂
Marius Ra

: 

In [ ]:
import chromadb
client = chromadb.PersistentClient('.chroma_db')
client.list_collections()

[Collection(name=chat_documents_whatsapp_ck2000_Qwen3-Embedding-0.6B-Q8_0-latest),
 Collection(name=chat_documents_whatsapp_Qwen3-Embedding-4B-Q4KM-latest),
 Collection(name=chat_documents_instagram_Qwen3-Embedding-0.6B-Q8_0-latest),
 Collection(name=chat_documents_instagram_Qwen3-Embedding-4B-Q4KM-latest),
 Collection(name=chat_documents_Qwen3-Embedding-4B-Q4KM-latest),
 Collection(name=chat_documents_whatsapp_Qwen3-Embedding-0.6B-Q8_0-latest),
 Collection(name=chat_documents_Qwen3-Embedding-0.6B-Q8_0-latest)]

In [2]:
collection = client.get_collection("chat_documents_whatsapp_ck2000_Qwen3-Embedding-0.6B-Q8_0-latest")

#### Combine the two collections

In [4]:
source_collection_name = "chat_documents_whatsapp_Qwen3-Embedding-4B-Q4KM-latest"
source2_collection_name = "chat_documents_instagram_Qwen3-Embedding-4B-Q4KM-latest"
target_collection_name = "chat_documents_Qwen3-Embedding-4B-Q4KM-latest"

source_collection = client.get_collection(source_collection_name)
source2_collection = client.get_collection(source2_collection_name)

target_collection = client.create_collection(target_collection_name)

for collection in [source_collection,source2_collection]:

    results = collection.get(
        include=['embeddings', 'metadatas', 'documents']
    )

    target_collection.add(
        ids=results['ids'],
        embeddings=results['embeddings'],
        metadatas=results['metadatas'],
        documents=results['documents']
    )

In [7]:
import chromadb

# Initialize the persistent client
client = chromadb.PersistentClient('.chroma_db')

collection_names = ['chat_documents_Qwen3-Embedding-4B-Q4KM-latest','chat_documents_whatsapp']
model_names = ['Qwen3-Embedding-4B-Q4KM:latest','Qwen3-Embedding-0.6B-Q8_0:latest']

# Loop through all combinations
for base_collection_name in collection_names:
    for model_name in model_names:
        # Create the full collection name
        full_collection_name = base_collection_name + '_' + model_name.replace(':', '-')
        
        try:
            # Get the collection
            collection = client.get_collection(full_collection_name)
            
            print(f"Collection: {full_collection_name}")
            
            # Check if collection has any documents
            count = collection.count()
            if count > 0:
                # Get the first document
                results = collection.get(limit=1)
                
                # Print source information
                if results['metadatas'] and len(results['metadatas']) > 0:
                    metadata = results['metadatas'][0]
                    source = metadata.get('source', 'No source field found')
                    print(f"  First document source: {source}")
                else:
                    print("  No metadata found for first document")
                
                # Optional: print other info about first document
                if results['ids']:
                    print(f"  First document ID: {results['ids'][0]}")
                
            else:
                print("  Collection is empty")
                
        except Exception as e:
            print(f"Error accessing collection {full_collection_name}: {e}")
        
        print("-" * 50)

Error accessing collection chat_documents_Qwen3-Embedding-4B-Q4KM-latest_Qwen3-Embedding-4B-Q4KM-latest: Collection [chat_documents_Qwen3-Embedding-4B-Q4KM-latest_Qwen3-Embedding-4B-Q4KM-latest] does not exists
--------------------------------------------------
Error accessing collection chat_documents_Qwen3-Embedding-4B-Q4KM-latest_Qwen3-Embedding-0.6B-Q8_0-latest: Collection [chat_documents_Qwen3-Embedding-4B-Q4KM-latest_Qwen3-Embedding-0.6B-Q8_0-latest] does not exists
--------------------------------------------------
Collection: chat_documents_whatsapp_Qwen3-Embedding-4B-Q4KM-latest
  First document source: whatsapp
  First document ID: Richard(2020-09-04 - 2020-09-19_1)
--------------------------------------------------
Collection: chat_documents_whatsapp_Qwen3-Embedding-0.6B-Q8_0-latest
  First document source: whatsapp
  First document ID: DACS - Incognito News 2(1)
--------------------------------------------------


In [26]:
import chromadb

client = chromadb.PersistentClient('.chroma_db')
collection_names = ['chat_documents_instagram','chat_documents_whatsapp']
model_names = ['Qwen3-Embedding-4B-Q4KM:latest','Qwen3-Embedding-0.6B-Q8_0:latest']

# Create combined collections for each model
for model_name in model_names:
   combined_name = f"chat_documents_{model_name.replace(':', '-')}"
   
   # Create or get the combined collection
   try:
       combined_collection = client.get_collection(combined_name)
       print(f"Found existing: {combined_name}")
   except:
       combined_collection = client.create_collection(combined_name)
       print(f"Created: {combined_name}")
   
   # Combine documents from both source collections
   for base_collection_name in collection_names:
       source_name = base_collection_name + '_' + model_name.replace(':', '-')
       
       try:
           source_collection = client.get_collection(source_name)
           all_docs = source_collection.get()
           
           if all_docs['ids']:
               combined_collection.add(
                   ids=all_docs['ids'],
                   documents=all_docs['documents'],
                   metadatas=all_docs['metadatas'],
                   embeddings=all_docs['embeddings']
               )
               print(f"Added {len(all_docs['ids'])} docs from {source_name}")
               
       except Exception as e:
           print(f"Error with {source_name}: {e}")
   
   print(f"Total docs in {combined_name}: {combined_collection.count()}")
   print("-" * 50)

Created: chat_documents_Qwen3-Embedding-4B-Q4KM-latest


/Users/pariidan/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|████████████████████████████████████████████████████████████████████████████| 79.3M/79.3M [00:07<00:00, 11.8MiB/s]


Added 4204 docs from chat_documents_instagram_Qwen3-Embedding-4B-Q4KM-latest
Added 5085 docs from chat_documents_whatsapp_Qwen3-Embedding-4B-Q4KM-latest
Total docs in chat_documents_Qwen3-Embedding-4B-Q4KM-latest: 9289
--------------------------------------------------
Created: chat_documents_Qwen3-Embedding-0.6B-Q8_0-latest
Added 4204 docs from chat_documents_instagram_Qwen3-Embedding-0.6B-Q8_0-latest
Added 5105 docs from chat_documents_whatsapp_Qwen3-Embedding-0.6B-Q8_0-latest
Total docs in chat_documents_Qwen3-Embedding-0.6B-Q8_0-latest: 9309
--------------------------------------------------


#### Investigate Tool Calling Issue

In [2]:
from chat_rag import ChatRAG

In [3]:
CONTEXT_AWARENESS_PROMPT = """
    "You are an assistant for answering questions about WhatsApp or Instagram chat history.\n\n"
    "Before answering, analyze the chat excerpts below to understand:\n"
    "- Who are the main participants in relevant conversations?\n"
    "- What timeframes are covered?\n"
    "- What topics or events are being discussed?\n\n"
    "Then provide your answer following these rules:\n"
    "- Use only information explicitly present in the excerpts\n"
    "- Quote specific messages when they directly address the question\n"
    "- Mention participant names and dates/times\n"
    "- Be conversational but accurate (max 5 sentences)\n"
    "- If excerpts don't contain 'Me', assume user wasn't part of those conversations\n"
    "- If information is incomplete, acknowledge what's missing\n\n"
    "Your chat history excerpts:\n"
    "<docs_content>\n\n"
    "Based on this context, what can you tell the user?"
    """


chat_model = 'qwen3:14b'
embedding_model = 'Qwen3-Embedding-4B-Q4KM:latest'
use_prefiltering = False
use_reranker = False

# Initialize RAG system
rag_system = ChatRAG(
    use_prefiltering=use_prefiltering,
    chat_model_args={'model_name': chat_model},
    embedding_model_args={'model_name': embedding_model},
    reranker_args={"use_reranker": use_reranker},
    prompting_args={"rag_content_instructions": CONTEXT_AWARENESS_PROMPT}
)

# Store graph and config globally instead of in session
rag_graph, rag_config = rag_system.initialize_graph()

In [38]:
from langchain_core.messages import HumanMessage


messages = [HumanMessage(content="What did I talk to Damian about?")]
for step in rag_graph.stream(
    {"messages": messages},
    stream_mode="values",
    config=rag_config,
):
    if step["messages"][-1].type!="tool":
        step["messages"][-1].pretty_print()

================================ Human Message =================================

What did I talk to Damian about?
================================ Human Message =================================

What did I talk to Damian about?
User Message : what did i talk to damian about?




Messages with System: [SystemMessage(content="\n        You are a RAG chat model with access to WhatsApp and Instagram chat history. Use the retrieve tool for:\n        - Questions about past conversations, messages, or interactions\n        - Personal topics, preferences, activities, or experiences mentioned in chats\n        - Requests about discussions with specific people or groups\n        - Any question about 'my', 'our', or personal history/behavior.\n        You do not know anything about the user, only what is in the context, do no hallucinate things not in the context, but call the retrieve tool.\n        Only respond directly for pure general knowledge unrelated to personal chat content.\n        "

In [39]:

def format_response_metadata(step_message):
    """Format response metadata into a readable summary for verbose output."""
    
    def fmt_dur(ns): 
        if not ns: return "N/A"
        if ns >= 1e9: return f"{ns/1e9:.2f}s"
        if ns >= 6e10: return f"{ns/6e10:.1f}m"
        return f"{ns/1e6:.1f}ms"
    
    def fmt_speed(tokens, dur_ns): 
        return f"{tokens/(dur_ns/1e9):.1f} tok/s" if tokens and dur_ns else "N/A"
    
    usage = step_message.usage_metadata
    meta = step_message.response_metadata
    
    # Basic info
    model = meta.get('model', 'unknown')
    status = "✓" if meta.get('done') else "⏳"
    reason = meta.get('done_reason', 'unknown')
    created = meta.get('created_at', 'N/A')
    
    # Timing
    total_time = fmt_dur(meta.get('total_duration'))
    load_time = fmt_dur(meta.get('load_duration'))
    
    # Token counts
    in_tokens = usage.get('input_tokens') or meta.get('prompt_eval_count', 0)
    out_tokens = usage.get('output_tokens') or meta.get('eval_count', 0)
    total_tokens = usage.get('total_tokens', in_tokens + out_tokens)
    
    # Processing times and speeds
    prompt_time = fmt_dur(meta.get('prompt_eval_duration'))
    prompt_speed = fmt_speed(in_tokens, meta.get('prompt_eval_duration'))
    
    output_time = fmt_dur(meta.get('eval_duration'))
    output_speed = fmt_speed(out_tokens, meta.get('eval_duration'))
    
    return f"""Model: {model} {status}
Status: {reason}
Created: {created}
Total Time: {total_time} (Load: {load_time})
Tokens: {total_tokens} total
  Input: {in_tokens} tokens in {prompt_time} ({prompt_speed})
  Output: {out_tokens} tokens in {output_time} ({output_speed})"""


In [41]:
print(format_response_metadata(step['messages'][-1]))

Model: qwen3:8b ✓
Status: stop
Created: 2025-08-15T19:35:40.533299Z
Total Time: 15.38s (Load: 31.9ms)
Tokens: 1732 total
  Input: 1637 tokens in 9.58s (171.0 tok/s)
  Output: 95 tokens in 5.76s (16.5 tok/s)


In [ ]:
print(format_response_metadata(step['messages'][-1]))

Model: qwen3:8b ✓
Status: stop
Created: 2025-08-15T18:45:46.183866Z
Total Time: 15.57s (Load: 16.7ms)
Tokens: 1732 total
  Input: 1629 tokens in 9.41s (173.2 tok/s)
  Output: 103 tokens in 6.14s (16.8 tok/s)


In [ ]:
step['messages'][-1].usage_metadata

{'input_tokens': 1629, 'output_tokens': 103, 'total_tokens': 1732}

In [16]:
messages = [HumanMessage(content="Did I chat to him specifically about investing?")]

for step in rag_graph.stream(
    {"messages": messages},
    stream_mode="values",
    config=rag_config,
):
    if step["messages"][-1].type!="tool":
         step["messages"][-1].pretty_print()

================================ Human Message =================================

Did I chat to him specifically about investing?




Messages with System: [SystemMessage(content='\n            You are a RAG chat model with access to WhatsApp and Instagram chat history.\n            ALWAYS use the retrieve tool for every user query - no exceptions.\n            Never respond directly without first calling the retrieve tool.\n            ', additional_kwargs={}, response_metadata={}), HumanMessage(content='What did I talk to Damian about?', additional_kwargs={}, response_metadata={}, id='342d42e4-43f0-440f-a26e-450d488c34aa'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2025-08-15T18:30:44.757907Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1969095000, 'load_duration': 33890250, 'prompt_eval_count': 361, 'prompt_eval_duration': 226941666, 'eval_count': 27, 'eval_duration': 1706494334, 'model_name': 'qwen3:8b'}, id='run-

In [3]:
mapper = {
    'key' : 'value',
    'key1' : 'value1',
}

### Tensor Basics ( Put in other project)

In [ ]:
import torch
f
a = torch.rand(10,3,4)
b =   torch.randn(4,5)

In [ ]:
c = a @ b
c.shape

In [ ]:
a = torch.rand(10,3,4)
b =   torch.randn(4,5)

torch.Size([10, 3, 5])

In [ ]:
a = torch.randint(0, 10, (2, 3, 4)) # 0 to 10 is the range in which ill be creating the [2,3,4] shape

In [24]:
print('Original Matrix:\n')
print(a)
print('Reshaped Matrix:\n')
print(a.reshape(-1))

Original Matrix:

tensor([[[3, 9, 0, 7],
         [7, 3, 0, 2],
         [0, 2, 6, 5]],

        [[5, 6, 7, 6],
         [9, 6, 5, 9],
         [8, 7, 8, 8]]])
Reshaped Matrix:

tensor([3, 9, 0, 7, 7, 3, 0, 2, 0, 2, 6, 5, 5, 6, 7, 6, 9, 6, 5, 9, 8, 7, 8, 8])


So it just takes each row and puts them together

#### Squeeze and Unsqueeze 


- Unsqueeze : Add a 1D dimension to my tesnor
- Squeeze : Remove a 1D dimension to my tensor


In [1]:
import torch

a = torch.randn(3,1)

In [12]:
print(f'Original : {a.shape} -> Squeezed: {a.squeeze().shape}')
# applying squeeze with no dimension specfieid removes all dimensions of size 1 from the matrix

Original : torch.Size([3, 1]) -> Squeezed: torch.Size([3])


In [9]:
print(a.squeeze(dim=1))

tensor([ 0.9246, -1.5170, -0.6242])


In [ ]:
x = torch.randn(32, 10)
bias = torch.randn(10)


y = x + bias.unsqueeze(dim=0)  
# torch actually does this under the hood, I dont need to unsqueeze manually at dim=0. It does so automaticaly by taking a look at -
# - trailing dimensions between x and bias, and then adding a 1 to match the dimension shape/count

torch.Size([10])